# SuAVE Notebook Dispatcher

**Binder**: survey parameters were loaded automatically — just run all cells.

**Colab**: Google may ask you to sign in and authorize the notebook before it runs (only if you haven't authorized it before). Once past that, paste the session token into `SUAVE_TOKEN` and your SuAVE server URL into `SUAVE_HOST`, then run all cells.

**Colab only** — to avoid re-entering credentials in each operation notebook, run the cell below to mount Google Drive before loading parameters.


In [ ]:
# ── Setup: import suave_utils (all environments) ──────────────────────────
import os, sys

# In Colab the repo must be cloned first (only the .ipynb file is fetched)
if "google.colab" in sys.modules and not os.path.isfile("helpers/suave_utils.py"):
    import subprocess
    _r = subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/izaslavsky/suave-notebooks.git", "/tmp/suave-nb"],
        capture_output=True, text=True
    )
    if _r.returncode != 0:
        raise RuntimeError(f"Could not clone suave-notebooks repo:\n{_r.stderr}")
    os.chdir("/tmp/suave-nb")

sys.path.insert(0, 'helpers')
import suave_utils as su
from IPython.display import display, HTML


In [ ]:
# ── Colab only: mount Google Drive to persist credentials across notebooks ──
if su.ENV.colab:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    su._skipped('this cell is for Colab only')


In [ ]:
# ── Binder / JupyterHub / local: parameters load automatically ─────────────
if not su.ENV.colab:
    params = su.load_params()
    if params:
        su.show_params(params)
else:
    su._skipped('this cell is for Binder / JupyterHub / local only')


In [ ]:
# ── Colab: load from Drive if a session exists there ──────────────────────
if su.ENV.colab:
    params = su.load_params(_silent=True)
    if params:
        su.show_params(params)
        display(HTML(
            '<p style="color:#6b7280;font-size:12px;margin:4px 0">'
            'Cached session found. If this is the wrong survey, paste a new token '
            'in the cell below and re-run it.</p>'))
else:
    su._skipped('this cell is for Colab only')


In [ ]:
# ── Colab: enter token to load (or switch to) a survey ────────────────────
# Always runs. Leave blank to keep the session loaded above.
# Paste a new token here whenever you want to switch to a different survey.
if su.ENV.colab:
    SUAVE_TOKEN = ''   # @param {type:"string"}
    SUAVE_HOST  = ''   # @param {type:"string"}
    if SUAVE_TOKEN.strip():
        # Token provided -- always fetch fresh, overwrites Drive cache
        params = su.load_params(token=SUAVE_TOKEN.strip(), host=SUAVE_HOST.strip())
        if params:
            su.show_params(params)
    elif params is not None:
        su._skipped('no token entered -- keeping session loaded above')
    else:
        display(HTML(
            '<p style="color:#e07000">No session loaded. '
            'Paste SUAVE_TOKEN and SUAVE_HOST above and re-run this cell.</p>'))
else:
    su._skipped('this cell is for Colab only')


In [ ]:
# ── Cell 2: Choose an operation ────────────────────────────────────────────
from IPython.display import display, HTML

if params is None:
    raise RuntimeError('Run the parameter-loading cell above first (Cell 3 on Binder, Cell 4 on Colab).')

# ── Detect what this survey and hub support ──────────────────────────────────
caps             = su.detect_capabilities(params)
has_images       = caps['has_images']
has_netvis       = caps['has_netvis']
has_largedataset = caps['has_largedataset']
localdzc         = caps['localdzc']
full_images      = caps['full_images']

# ── Menu definition ──────────────────────────────────────────────────────────
# Availability keys:
#   'any'         — always shown
#   'image'       — full-size images accessible on NFS (/lib-nfs/dzgen/...)
#   'netvis'      — survey has a #netvis network data file
#   'largedataset'— /lib-nfs/largedatasets is mounted on this hub

SECTIONS = [
    ('Data & Statistics', '\U0001f4ca', '#6366f1', [
        ('Descriptive Statistics',      'operations/stats/DescriptiveStats.ipynb',              'any'),
        ('Contingency Tables',          'operations/stats/Generate_Contingency_Tables.ipynb',   'any'),
        ('Factor Contributions',        'operations/stats/Generate_Factor_Contributions.ipynb', 'any'),
        ('Arithmetic Operations',       'operations/arithmetic/SuaveArithmetic.ipynb',          'any'),
        ('Spatial Statistics',          'operations/spatialstats/SpatialStats.ipynb',           'any'),
        ('Generate Maps',               'operations/maps/Generate_Aggregate_Maps_Suave.ipynb',  'any'),
        ('Network Analysis',            'operations/networks/networks.ipynb',                   'netvis'),
    ]),
    ('ML & Statistical Modeling', '\U0001f9e0', '#7c3aed', [
        ('Classification',              'operations/stats/SupervisedClassification.ipynb',      'any'),
        ('Regression',                  'operations/stats/SupervisedRegression.ipynb',          'any'),
    ]),
    ('Unsupervised & Dimensionality', '\U0001f4d0', '#0d9488', [
        ('PCA / Dimensionality',        'operations/stats/PCA.ipynb',                          'any'),
        ('Clustering',                  'operations/stats/Clustering.ipynb',                   'any'),
        ('Outlier Detection',           'operations/stats/OutlierDetection.ipynb',             'any'),
        ('Network Metrics',             'operations/networks/NetworkMetrics.ipynb',            'netvis'),
        ('Variable Transforms',         'operations/wrangling/VariableTransforms.ipynb',       'any'),
    ]),
    ('Text & NLP', '\U0001f4dd', '#0ea5e9', [
        ('SDG Dataset',                 'operations/SDG/GenerateSDGDataset.ipynb',              'largedataset'),
    ]),
    ('Image Analysis', '\U0001f5bc️', '#f59e0b', [
        ('Color Statistics',            'operations/colors/ColorStats.ipynb',                  'image'),
        ('Classify Images (Clarifai)',  'operations/classify/ImageClassify.ipynb',             'image'),
        ('Transfer Learning',           'operations/transfer_learning/transfer_learning.ipynb','image'),
    ]),
    ('AI — Text', '\U0001f916', '#8b5cf6', [
        ('Sentiment Analysis',        'operations/ai_text/SentimentAnalysis.ipynb',    'any'),
        ('Zero-Shot Classification',  'operations/ai_text/ZeroShotClassify.ipynb',     'any'),
        ('Geocoding from Text',       'operations/ai_text/Geocoding.ipynb',            'any'),
        ('Topic Modeling',            'operations/ai_text/TopicModeling.ipynb',        'any'),
        ('Entity Linking (Wikidata)', 'operations/ai_text/EntityLinking.ipynb',        'any'),
        ('OpenAlex Enrichment',       'operations/ai_text/OpenAlexEnrich.ipynb',       'any'),
    ]),
    ('AI — Images', '\U0001f50d', '#ec4899', [
        ('Image Captioning (BLIP)',   'operations/ai_image/ImageCaptioning.ipynb',     'image'),
        ('Object Detection (DETR)',   'operations/ai_image/ObjectDetection.ipynb',     'image'),
        ('Image Clustering (CLIP)',   'operations/ai_image/ImageClustering.ipynb',     'image'),
    ]),
]
def _avail(kind):
    return (
        kind == 'any'
        or (kind == 'image'        and has_images)
        or (kind == 'netvis'       and has_netvis)
        or (kind == 'largedataset' and has_largedataset)
    )

# ── Build warning banners ────────────────────────────────────────────────────
import os
warnings = []
if localdzc and os.path.isfile(localdzc) and not os.path.isdir(full_images):
    warnings.append(
        'This hub supports image-based processing, but full-size images have not '
        'been generated for this survey. Contact the admin to re-generate images '
        'from image tiles. Image analysis operations are unavailable.'
    )
elif not localdzc:
    warnings.append(
        'Image tiles are not on NFS-mounted storage for this survey '
        '(or this hub does not support NFS). Image analysis operations are unavailable.'
    )
if not has_largedataset:
    warnings.append(
        'Large external datasets (e.g. SDG) are not mounted on this hub '
        '(/lib-nfs/largedatasets). Those operations are unavailable.'
    )

# ── Build HTML card menu ─────────────────────────────────────────────────────
parts = ['''
<style>
  .sd-wrap { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
              max-width: 780px; padding: 6px 0 12px; }
  .sd-warn { padding: 9px 13px; border-radius: 6px; font-size: 12px; line-height: 1.5;
              margin-bottom: 10px; background: #fffbeb; border: 1px solid #f59e0b;
              color: #92400e; }
  .sd-section { margin: 18px 0 6px; }
  .sd-section-hd { display: flex; align-items: center; gap: 7px;
                    font-size: 11px; font-weight: 700; text-transform: uppercase;
                    letter-spacing: .07em; margin-bottom: 9px;
                    padding-bottom: 6px; border-bottom: 2px solid; }
  .sd-cards { display: flex; flex-wrap: wrap; gap: 7px; }
  .sd-card { display: inline-block; padding: 7px 14px; border-radius: 7px;
              border: 1.5px solid; font-size: 13px; font-weight: 500;
              text-decoration: none !important; line-height: 1.3;
              transition: filter .12s, transform .12s; }
  .sd-card:hover { filter: brightness(1.18); transform: translateY(-1px);
                    text-decoration: none !important; }
  .sd-note { margin-top: 20px; font-size: 11px; color: #6b7280; line-height: 1.7;
              padding: 10px 14px; border-radius: 6px; background: #f9fafb;
              border: 1px solid #e5e7eb; }
</style>
<div class="sd-wrap">
''']

for w in warnings:
    parts.append(f'<div class="sd-warn">&#9888;&nbsp; {w}</div>')

any_section = False
for title, icon, color, items in SECTIONS:
    visible = [(lbl, path) for lbl, path, kind in items if _avail(kind)]
    if not visible:
        continue
    any_section = True
    bg     = color + '18'
    border = color + '60'
    hdbdr  = color + '50'
    parts.append(
        f'<div class="sd-section">'
        f'<div class="sd-section-hd" style="color:{color};border-color:{hdbdr}">'
        f'<span>{icon}</span><span>{title}</span></div>'
        f'<div class="sd-cards">'
    )
    for lbl, path in visible:
        url = su.make_nb_url(path)
        parts.append(
            f'<a class="sd-card" href="{url}" target="_blank" '
            f'style="color:{color};background:{bg};border-color:{border}">'
            f'{lbl}</a>'
        )
    parts.append('</div></div>')

if not any_section:
    parts.append('<p style="color:#6b7280">No operations available for this survey.</p>')

if su.in_colab():
    import pathlib as _pl
    _drive_ok = _pl.Path('/content/drive/MyDrive/.suave_params.json').exists()
    colab_note = (
        ' Parameters are saved to Google Drive and will load automatically in each notebook.'
        if _drive_ok else
        ' Each notebook opens in a new Colab session — you will need to enter '
        'SUAVE_TOKEN and SUAVE_HOST, or mount Google Drive before running this notebook.'
    )
else:
    colab_note = ''
parts.append(
    f'<div class="sd-note">'
    f'↗️ Each notebook opens in a new tab and reads the same survey '
    f'parameters automatically — no token required.{colab_note}'
    f'</div>'
)
parts.append('</div>')

display(HTML(''.join(parts)))